# Predicting Food Delivery Time with Machine Learning

When someone places a food order, the delivery estimate is one of the first things they want to know. Getting that estimate right is important: a prediction that is too optimistic can be frustrating, while one that is too cautious may not be useful either.

In this project, I use historical DoorDash order data to predict the time between an order being placed and the moment it reaches the customer. I start by looking at the available data, build a few additional features, and work through the usual preprocessing steps before comparing several regression models.

The solution takes an interesting turn toward the end. Predicting the full delivery duration directly gives fairly high errors, so I also try breaking the problem into parts. Instead of predicting the entire duration in one go, I estimate the preparation time first and then use that estimate alongside the known order placement and driving durations.

The notebook follows that investigation from the first look at the data through to the final model comparison.

## The task

For each order, predict the total delivery duration in seconds:

- **Start:** when the customer submits the order (`created_at`)
- **End:** when the order is delivered (`actual_delivery_time`)

The target is therefore the difference between these two timestamps.

## Notebook roadmap

1. Load the data and explore the available columns.
2. Create the delivery duration target and a few useful features.
3. Encode categorical variables and prepare the modelling table.
4. Look for redundant features and multicollinearity.
5. Select features using a Random Forest and inspect PCA.
6. Compare classical regression models with different feature sets and scaling choices.
7. Try a two stage approach that predicts preparation time first.
8. Compare the final regression stage with a small neural network.

**Before running the notebook:** place `historical_data.csv` in the same folder as this notebook. Some of the model comparisons can take a while, especially the Random Forest runs.

## 1. Bring in the libraries

I'll use Pandas and NumPy for working with the data, Matplotlib and Seaborn for quick visual checks, and scikit-learn for preprocessing and the first set of models.

The solution also compares XGBoost and LightGBM, uses Statsmodels for the VIF calculation, and finishes with a small TensorFlow/Keras network.

In [ ]:
# If you need to install the project dependencies, run:
# %pip install pandas numpy matplotlib seaborn scikit-learn statsmodels xgboost lightgbm tensorflow

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn import linear_model, tree
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import MinMaxScaler, StandardScaler

from statsmodels.stats.outliers_influence import variance_inflation_factor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

np.random.seed(42)
sns.set_theme(style="whitegrid")

print("Libraries imported successfully.")

## 2. Load the historical orders

The file contains close to 200,000 historical deliveries. First, I'll take a look at a few rows and check the data types and missing values.

One thing to notice straight away is that the target we need is not provided as a separate column. It has to be calculated from the order and delivery timestamps.

In [ ]:
historical_data = pd.read_csv("historical_data.csv")

display(historical_data.head())
print(f"Rows: {historical_data.shape[0]:,}")
print(f"Columns: {historical_data.shape[1]}")

historical_data.info()

## 3. Create the delivery duration target

Both timestamp columns are currently stored as text. I'll convert them to datetime values and subtract the order time from the delivery time.

That gives us the actual delivery duration in seconds, which is the value the regression models will try to predict.

In [ ]:
historical_data["created_at"] = pd.to_datetime(historical_data["created_at"])
historical_data["actual_delivery_time"] = pd.to_datetime(historical_data["actual_delivery_time"])

historical_data["actual_total_delivery_duration"] = (
    historical_data["actual_delivery_time"] - historical_data["created_at"]
).dt.total_seconds()

display(
    historical_data[[
        "created_at",
        "actual_delivery_time",
        "actual_total_delivery_duration"
    ]].head()
)

historical_data["actual_total_delivery_duration"].describe()

## 4. Add a couple of operational features

Before getting into encoding and model selection, I'll create two features from columns that are already in the dataset.

The first is an estimate of the non preparation part of the delivery. It combines the estimated order placement duration with the estimated drive from the store to the customer.

The second is the busy dasher ratio. It gives us a rough view of how busy the delivery operation is by comparing the number of busy dashers with the number currently on shift.

A ratio above 1 appears in some rows. Rather than assuming those records are wrong, I'll leave them for now and inspect how they behave during cleaning.

In [ ]:
historical_data["estimated_non_prep_duration"] = (
    historical_data["estimated_store_to_consumer_driving_duration"]
    + historical_data["estimated_order_place_duration"]
)

historical_data["busy_dashers_ratio"] = (
    historical_data["total_busy_dashers"]
    / historical_data["total_onshift_dashers"]
)

display(historical_data[[
    "estimated_order_place_duration",
    "estimated_store_to_consumer_driving_duration",
    "estimated_non_prep_duration",
    "total_onshift_dashers",
    "total_busy_dashers",
    "busy_dashers_ratio"
]].head())

# Have a look at cases where the ratio is above one.
display(
    historical_data.loc[
        historical_data["busy_dashers_ratio"] > 1,
        ["total_onshift_dashers", "total_busy_dashers", "busy_dashers_ratio"]
    ].head()
)

## 5. Decide how to handle the categorical columns

There are two identifier columns worth looking at: `market_id` and `store_id`. There are only six markets, but more than 6,700 stores. One hot encoding every store would add a very large number of columns, so I'll keep the market information for now and leave `store_id` out of the model.

`order_protocol` has seven distinct values, so it is another candidate for one hot encoding.

The store category column has missing values. The original solution fills a missing category using the most frequent category associated with that store. There is a small wrinkle here: some stores appear with more than one category, so a tie can occur. Pandas returns the tied modes in order, and the first one is used for filling.

In [ ]:
print("Unique markets:", historical_data["market_id"].nunique())
print("Unique stores:", historical_data["store_id"].nunique())
print("Order protocols:", historical_data["order_protocol"].nunique())
print("Store categories:", historical_data["store_primary_category"].nunique())

In [ ]:
# Build a lookup of the most common category for each store.
store_id_unique = historical_data["store_id"].dropna().unique().tolist()

store_id_and_category = {
    store_id: historical_data.loc[
        historical_data["store_id"] == store_id,
        "store_primary_category"
    ].mode()
    for store_id in store_id_unique
}

def fill_store_category(store_id):
    """Use the first most frequent category recorded for this store."""
    category_modes = store_id_and_category.get(store_id, pd.Series(dtype=object))
    if len(category_modes) == 0:
        return np.nan
    return category_modes.iloc[0]

historical_data["nan_free_store_primary_category"] = (
    historical_data["store_id"].apply(fill_store_category)
)

print(
    "Missing categories before:",
    historical_data["store_primary_category"].isna().sum()
)
print(
    "Missing categories after:",
    historical_data["nan_free_store_primary_category"].isna().sum()
)

# Check a store that appears with more than one recorded category.
display(store_id_and_category.get(5477))

The store level lookup is useful, but it is not perfect. For example, store `5477` appears with both Mexican and Indian categories. When the modes are tied, the fill function uses the first value returned. That is worth remembering when interpreting the category features later.

I'll now one hot encode the market, order protocol and store category fields. The original identifier and text columns can then be removed from the modelling table.

In [ ]:
order_protocol_dummies = pd.get_dummies(
    historical_data["order_protocol"],
    prefix="order_protocol"
)

market_id_dummies = pd.get_dummies(
    historical_data["market_id"],
    prefix="market_id"
)

store_primary_category_dummies = pd.get_dummies(
    historical_data["nan_free_store_primary_category"],
    prefix="category"
)

print("Order protocol dummy columns:", order_protocol_dummies.shape[1])
print("Market dummy columns:", market_id_dummies.shape[1])
print("Store category dummy columns:", store_primary_category_dummies.shape[1])

## 6. Assemble a numeric modelling table

The timestamp columns are no longer needed as predictors because the target has already been calculated. The original categorical columns can also be dropped because their encoded versions are ready.

For this first pass, I'll combine the numerical features with all three groups of dummy variables. The table is converted to `float32` so that the modelling libraries all receive a consistent numeric format.

In [ ]:
train_df = historical_data.drop(columns=[
    "created_at",
    "market_id",
    "store_id",
    "store_primary_category",
    "actual_delivery_time",
    "nan_free_store_primary_category",
    "order_protocol"
])

train_df = pd.concat(
    [
        train_df,
        order_protocol_dummies,
        market_id_dummies,
        store_primary_category_dummies
    ],
    axis=1
)

train_df = train_df.astype("float32")

print("Initial modelling table:", train_df.shape)
display(train_df.head())
display(train_df.describe().T.head(15))

A quick summary shows that some columns contain missing values and the busy dasher ratio includes infinite values. Those infinities can happen when the number of dashers on shift is zero.

For this version of the solution, I'll replace positive and negative infinity with missing values and drop rows that still contain missing values. This leaves a clean numeric table for the feature analysis.

In [ ]:
train_df.replace([np.inf, -np.inf], np.nan, inplace=True)
train_df.dropna(inplace=True)

print("Shape after removing rows with missing or infinite values:", train_df.shape)
print("Remaining missing values:", int(train_df.isna().sum().sum()))
print("All values finite:", bool(np.isfinite(train_df.to_numpy()).all()))

## 7. Look for redundant features

With the categorical variables encoded, the table has a fairly large number of columns. Some of them may carry almost the same information.

I'll start with a correlation matrix. Since the matrix is symmetrical, the heatmap only needs to show one triangle. I'll also create a small helper to list the strongest absolute correlations between pairs of features.

In [ ]:
def get_redundant_pairs(data):
    """Return the lower triangle and diagonal pairs of a correlation matrix."""
    pairs_to_drop = set()
    columns = data.columns

    for i in range(len(columns)):
        for j in range(i + 1):
            pairs_to_drop.add((columns[i], columns[j]))

    return pairs_to_drop


def get_top_abs_correlations(data, n=20):
    """List the strongest unique feature correlations."""
    correlations = data.corr().abs().unstack()
    correlations = correlations.drop(
        labels=get_redundant_pairs(data),
        errors="ignore"
    )
    return correlations.sort_values(ascending=False).head(n)

In [ ]:
corr = train_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

plt.figure(figsize=(13, 10))
cmap = sns.diverging_palette(230, 20, as_cmap=True)

sns.heatmap(
    corr,
    mask=mask,
    cmap=cmap,
    vmax=1,
    vmin=-1,
    center=0,
    square=True,
    linewidths=0.3,
    cbar_kws={"shrink": 0.6}
)

plt.title("Correlation between candidate features")
plt.tight_layout()
plt.show()

print("Strongest absolute correlations:")
display(get_top_abs_correlations(train_df, 20))

A few pairs stand out. The on shift and busy dasher counts are closely related, and the estimated non preparation duration is closely tied to its two ingredients. Some market and protocol dummy variables are also correlated with one another.

I'll simplify the feature set in stages rather than removing everything with a high correlation. For example, item counts and prices may still tell us something useful about preparation time, so I'll try representing them with more meaningful combinations.

In [ ]:
# Start again from the cleaned numerical and target columns.
train_df = historical_data.drop(columns=[
    "created_at",
    "market_id",
    "store_id",
    "store_primary_category",
    "actual_delivery_time",
    "nan_free_store_primary_category",
    "order_protocol"
])

# Keep the store category indicators, but leave out market and protocol dummies.
train_df = pd.concat(
    [train_df, store_primary_category_dummies],
    axis=1
)

# Remove features that are redundant with other columns.
train_df.drop(
    columns=[
        "total_onshift_dashers",
        "total_busy_dashers",
        "category_indonesian",
        "estimated_non_prep_duration"
    ],
    inplace=True,
    errors="ignore"
)

train_df = train_df.astype("float32")
train_df.replace([np.inf, -np.inf], np.nan, inplace=True)
train_df.dropna(inplace=True)

print("Shape after the first round of feature reduction:", train_df.shape)
display(get_top_abs_correlations(train_df, 20))

There is still a relationship between the number of items, the number of distinct items and the subtotal. Rather than dropping all of these columns, I'll derive a couple of ratios that describe the order a little more directly:

- the share of items that are distinct
- the average price per item

I'll also replace the minimum and maximum item prices with their range. The idea is to retain a useful summary while reducing overlap between the original columns.

In [ ]:
train_df["percent_distinct_item_of_total"] = (
    train_df["num_distinct_items"] / train_df["total_items"]
)

train_df["avg_price_per_item"] = (
    train_df["subtotal"] / train_df["total_items"]
)

train_df.drop(
    columns=["num_distinct_items", "subtotal"],
    inplace=True
)

train_df["price_range_of_items"] = (
    train_df["max_item_price"] - train_df["min_item_price"]
)

train_df.drop(
    columns=["max_item_price", "min_item_price"],
    inplace=True
)

print("Shape after feature engineering:", train_df.shape)
display(get_top_abs_correlations(train_df, 20))

The engineered features bring the strongest pairwise correlations down considerably. The distinct item percentage is still related to other variables, so I'll check multicollinearity formally in the next section.

At this point, the candidate table contains 82 columns, including the target.

In [ ]:
print("Current modelling table:", train_df.shape)
display(train_df.head())

## 8. Check multicollinearity with VIF

Correlation looks at two variables at a time. Variance Inflation Factor (VIF) goes a step further by checking how well one feature can be explained by the other features together.

I'll use a threshold of 20, as in the guided solution. To keep the calculation easier to follow, the loop removes only the feature with the highest VIF, recalculates the scores, and repeats until every remaining feature is below the threshold.

In [ ]:
def compute_vif(features):
    """Calculate VIF for the supplied predictor columns."""
    vif_data = pd.DataFrame()
    vif_data["feature"] = features
    vif_data["VIF"] = [
        variance_inflation_factor(train_df[features].values, i)
        for i in range(len(features))
    ]

    return vif_data.sort_values("VIF").reset_index(drop=True)


features = train_df.drop(
    columns=["actual_total_delivery_duration"]
).columns.tolist()

vif_data = compute_vif(features)
display(vif_data.tail(10))

In [ ]:
VIF_THRESHOLD = 20

while vif_data["VIF"].max() > VIF_THRESHOLD:
    highest_vif_feature = vif_data.iloc[-1]["feature"]
    print("Removing:", highest_vif_feature)

    features.remove(highest_vif_feature)
    vif_data = compute_vif(features)

selected_features = vif_data["feature"].tolist()

print(f"Remaining predictors: {len(selected_features)}")
display(vif_data.tail(10))

Only one feature needs to be removed to bring the remaining VIF values below the chosen threshold: `percent_distinct_item_of_total`.

I'll use the remaining columns as the starting feature set for model selection.

In [ ]:
X = train_df[selected_features]
y = train_df["actual_total_delivery_duration"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Predictor count:", X.shape[1])

## 9. Use a Random Forest to rank the features

The VIF check helps with multicollinearity, but it does not tell us which features are most useful for predicting delivery time.

For that, I'll fit a Random Forest Regressor and look at its feature importances. The ranking gives us a way to compare smaller feature sets with the full candidate set.

These importances are useful as a guide, but they should not be read as proof of cause and effect.

In [ ]:
forest = RandomForestRegressor(random_state=42)
forest.fit(X_train, y_train)

importances = pd.DataFrame(
    {"Gini-importance": forest.feature_importances_},
    index=X.columns
).sort_values("Gini-importance")

plt.figure(figsize=(14, 12))
importances["Gini-importance"].plot(kind="bar")
plt.title("Random Forest feature importance")
plt.ylabel("Importance")
plt.tight_layout()
plt.show()

plt.figure(figsize=(14, 9))
importances.tail(35)["Gini-importance"].plot(kind="bar")
plt.title("The 35 most important features")
plt.ylabel("Importance")
plt.tight_layout()
plt.show()

display(importances.tail(35).sort_values("Gini-importance", ascending=False))

The estimated store to customer driving duration is one of the strongest features, which makes intuitive sense. The busy dasher ratio, outstanding orders and average item price also contribute useful information.

I'll keep a few different feature set sizes for the model comparison: all available predictors, and the top 40, 20 and 10 features from the Random Forest ranking.

In [ ]:
feature_sets = {
    "full dataset": X.columns.tolist(),
    "selected_features_40": importances.tail(40).index.tolist(),
    "selected_features_20": importances.tail(20).index.tolist(),
    "selected_features_10": importances.tail(10).index.tolist()
}

for name, columns in feature_sets.items():
    print(f"{name}: {len(columns)} features")

## 10. Would PCA make the feature set smaller?

Principal Component Analysis is another way to reduce the number of dimensions. It combines the original variables into a smaller set of components that capture as much of the variation in the data as possible.

I'll standardize the training predictors, fit PCA, and plot the cumulative explained variance. This gives us a quick idea of how many components would be needed to retain most of the information.

In [ ]:
X_std = StandardScaler().fit_transform(X_train.values)

pca = PCA().fit(X_std)
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)

plt.figure(figsize=(9, 5))
plt.plot(np.arange(1, len(cumulative_variance) + 1), cumulative_variance)
plt.axhline(0.80, color="red", linestyle="--", label="80% variance")
plt.xlabel("Number of components")
plt.ylabel("Cumulative explained variance")
plt.title("PCA: cumulative explained variance")
plt.legend()
plt.tight_layout()
plt.show()

components_for_80 = np.argmax(cumulative_variance >= 0.80) + 1
print(f"Components needed to explain at least 80% of variance: {components_for_80}")

The original solution found that roughly 60 components were needed to explain 80% of the variance. That is not a particularly large reduction from the 80 predictors already available, so PCA is not used in the final modelling approach. The feature importance ranking gives us a more practical way to try smaller sets.

## 11. Set up scaling and the evaluation metric

The input features have very different ranges. For example, item counts are small numbers, while delivery durations and subtotals can be much larger. I'll compare StandardScaler and MinMaxScaler, as well as models trained without scaling.

The evaluation metric is Root Mean Squared Error (RMSE), measured in seconds. RMSE penalizes larger prediction errors more heavily than smaller ones, which is useful when a particularly late delivery estimate is especially undesirable.

The helper below scales the predictors and target separately, then another helper converts predictions back to seconds before calculating RMSE.

In [ ]:
from sklearn.base import clone

def scale_data(scaler, X, y):
    """Scale predictors and target, keeping a separate scaler for each."""
    X_scaler = clone(scaler)
    y_scaler = clone(scaler)

    X_scaled = X_scaler.fit_transform(X)
    y_scaled = y_scaler.fit_transform(y.to_numpy().reshape(-1, 1))

    return X_scaled, y_scaled, X_scaler, y_scaler


def rmse_with_inverse_transform(y_scaler, y_test, y_pred_scaled, model_name):
    """Return predictions and RMSE in the original delivery-time units."""
    y_predict = y_scaler.inverse_transform(
        np.asarray(y_pred_scaled).reshape(-1, 1)
    ).ravel()

    rmse = mean_squared_error(y_test, y_predict) ** 0.5
    print(f"{model_name} RMSE: {rmse:,.2f} seconds")

    return rmse, y_predict

## 12. Compare classical regression models

There is no obvious reason to expect one regressor to work best before trying them. I'll compare a mix of linear, tree based, boosting and neural network models:

- Ridge Regression
- Decision Tree
- Random Forest
- XGBoost
- LightGBM
- Scikit-learn's MLP Regressor

For each model, I'll try the four feature set sizes and three scaling options. The results will be collected in one table so that the comparisons are easy to inspect.

In [ ]:
def fit_and_score_regression(X_train, y_train, X_test, y_test, model):
    """Fit one regressor and return its predictions and train/test RMSE."""
    model.fit(X_train, y_train)

    train_predictions = model.predict(X_train)
    test_predictions = model.predict(X_test)

    train_rmse = mean_squared_error(y_train, train_predictions) ** 0.5
    test_rmse = mean_squared_error(y_test, test_predictions) ** 0.5

    return model, test_predictions, train_rmse, test_rmse


regression_models = {
    "Ridge": linear_model.Ridge(),
    "DecisionTree": tree.DecisionTreeRegressor(max_depth=6, random_state=42),
    "RandomForest": RandomForestRegressor(random_state=42),
    "XGBoost": XGBRegressor(random_state=42),
    "LGBM": LGBMRegressor(random_state=42),
    "MLP": MLPRegressor(random_state=42)
}

scalers = {
    "Standard scaler": StandardScaler(),
    "MinMax scaler": MinMaxScaler(),
    "NotScale": None
}

comparison_results = []

The next cell runs the full comparison. To avoid giving the models access to the test set while fitting the scalers, the train/test split is made first and each scaler is fitted on the training portion only.

Random Forest and boosting models can take a little while to finish across all combinations.

In [ ]:
for feature_set_name, columns in feature_sets.items():
    for scaler_name, scaler in scalers.items():
        X_subset = train_df[columns]
        y_target = train_df["actual_total_delivery_duration"]

        X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
            X_subset,
            y_target,
            test_size=0.2,
            random_state=42
        )

        if scaler is None:
            X_train_ready = X_train_raw
            X_test_ready = X_test_raw
            y_train_ready = y_train_raw
            y_test_ready = y_test_raw
            y_scaler = None
        else:
            X_scaler = clone(scaler)
            y_scaler = clone(scaler)

            X_train_ready = X_scaler.fit_transform(X_train_raw)
            X_test_ready = X_scaler.transform(X_test_raw)

            y_train_ready = y_scaler.fit_transform(
                y_train_raw.to_numpy().reshape(-1, 1)
            ).ravel()
            y_test_ready = y_scaler.transform(
                y_test_raw.to_numpy().reshape(-1, 1)
            ).ravel()

        for model_name, model_template in regression_models.items():
            model = clone(model_template)

            fitted_model, predictions, train_rmse, test_rmse = fit_and_score_regression(
                X_train_ready,
                y_train_ready,
                X_test_ready,
                y_test_ready,
                model
            )

            if y_scaler is not None:
                predictions = y_scaler.inverse_transform(
                    np.asarray(predictions).reshape(-1, 1)
                ).ravel()
                test_rmse = mean_squared_error(y_test_raw, predictions) ** 0.5
                train_predictions = y_scaler.inverse_transform(
                    fitted_model.predict(X_train_ready).reshape(-1, 1)
                ).ravel()
                train_rmse = mean_squared_error(y_train_raw, train_predictions) ** 0.5

            comparison_results.append({
                "regression_model": model_name,
                "feature_set": feature_set_name,
                "scaler_name": scaler_name,
                "train_RMSE": train_rmse,
                "RMSE": test_rmse
            })

        print(f"Finished {feature_set_name} with {scaler_name}.")

pred_df = pd.DataFrame(comparison_results)
display(pred_df.sort_values("RMSE").head(20))

Let's look at the test RMSE across the different combinations. Smaller values mean the predictions are, on average, closer to the actual delivery durations in the units used by the target: seconds.

In [ ]:
plt.figure(figsize=(14, 6))
sns.barplot(
    data=pred_df,
    x="regression_model",
    y="RMSE",
    hue="feature_set",
    errorbar=None
)
plt.title("Test RMSE across models and feature sets")
plt.ylabel("RMSE (seconds)")
plt.xlabel("Regression model")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

display(
    pred_df.sort_values("RMSE")
    .groupby(["feature_set", "scaler_name"], as_index=False)
    .first()
    .sort_values("RMSE")
)

The results from the original guided solution are close to one another across the model families, with LightGBM generally near the top. Reducing the feature set gives a small improvement at first, but removing too many predictors starts to hurt performance.

The original comparison reports a test RMSE of about **2,033 seconds** for LightGBM with the full feature set and StandardScaler, and roughly **2,035 seconds** with the selected 40 feature set. These are still sizeable errors, which led to the next idea: predict the preparation portion separately.

## 13. Try predicting preparation time instead

A delivery can be thought of as a few parts added together:

- order placement time
- food preparation time
- driving time from the store to the customer

The dataset already contains estimates for order placement and driving. What it does not contain is a direct estimate of the preparation time.

I'll create a new target by subtracting the two known estimates from the actual delivery duration. Then I'll train the model to predict this preparation component. The final duration can be reconstructed by adding the prediction back to the two estimates.

In [ ]:
train_df["prep_time"] = (
    train_df["actual_total_delivery_duration"]
    - train_df["estimated_store_to_consumer_driving_duration"]
    - train_df["estimated_order_place_duration"]
)

prep_feature_columns = feature_sets["selected_features_40"]

# These two estimates are added back after preparation time is predicted,
# so they are not included as predictors for the preparation model.
X_prep = train_df[prep_feature_columns].drop(
    columns=[
        "estimated_store_to_consumer_driving_duration",
        "estimated_order_place_duration"
    ],
    errors="ignore"
)
y_prep = train_df["prep_time"]

X_prep_train, X_prep_test, y_prep_train, y_prep_test = train_test_split(
    X_prep,
    y_prep,
    test_size=0.2,
    random_state=42
)

print("Preparation model predictors:", X_prep.shape[1])
print("Preparation target summary:")
display(y_prep.describe())

I'll use the same family of regressors to compare preparation time predictions. The guided solution found that LightGBM performed best in this stage, with an RMSE of approximately 2,036 seconds.

In [ ]:
prep_scaler_X = StandardScaler()
prep_scaler_y = StandardScaler()

X_prep_train_scaled = prep_scaler_X.fit_transform(X_prep_train)
X_prep_test_scaled = prep_scaler_X.transform(X_prep_test)

y_prep_train_scaled = prep_scaler_y.fit_transform(
    y_prep_train.to_numpy().reshape(-1, 1)
).ravel()
y_prep_test_scaled = prep_scaler_y.transform(
    y_prep_test.to_numpy().reshape(-1, 1)
).ravel()

prep_results = []

for model_name, model in prep_models.items():
    fitted_model, predictions_scaled, train_rmse_scaled, test_rmse_scaled = fit_and_score_regression(
        X_prep_train_scaled,
        y_prep_train_scaled,
        X_prep_test_scaled,
        y_prep_test_scaled,
        model
    )

    predictions = prep_scaler_y.inverse_transform(
        np.asarray(predictions_scaled).reshape(-1, 1)
    ).ravel()
    train_predictions = prep_scaler_y.inverse_transform(
        fitted_model.predict(X_prep_train_scaled).reshape(-1, 1)
    ).ravel()

    train_rmse = mean_squared_error(y_prep_train, train_predictions) ** 0.5
    test_rmse = mean_squared_error(y_prep_test, predictions) ** 0.5

    prep_results.append({
        "model": model_name,
        "train_RMSE": train_rmse,
        "test_RMSE": test_rmse
    })

display(pd.DataFrame(prep_results).sort_values("test_RMSE"))

For the next step, I'll use LightGBM as the preparation time model. I'll make predictions for the held out rows and keep the corresponding actual delivery durations and the two known time estimates together in one table.

In [ ]:
# Refit LightGBM and convert its preparation-time predictions back to seconds.
best_prep_model = LGBMRegressor(random_state=42)
best_prep_model.fit(X_prep_train_scaled, y_prep_train_scaled)

prep_duration_prediction_scaled = best_prep_model.predict(X_prep_test_scaled)
prep_duration_prediction = prep_scaler_y.inverse_transform(
    np.asarray(prep_duration_prediction_scaled).reshape(-1, 1)
).ravel()

values_df = pd.DataFrame({
    "actual_total_delivery_duration": train_df.loc[
        X_prep_test.index, "actual_total_delivery_duration"
    ].values,
    "prep_duration_prediction": prep_duration_prediction,
    "estimated_store_to_consumer_driving_duration": train_df.loc[
        X_prep_test.index, "estimated_store_to_consumer_driving_duration"
    ].values,
    "estimated_order_place_duration": train_df.loc[
        X_prep_test.index, "estimated_order_place_duration"
    ].values
}, index=X_prep_test.index)

display(values_df.head())

## 14. Rebuild total delivery time and check the error

As a first check, I'll simply add the predicted preparation duration to the two existing estimates.

This is the most direct way to reconstruct the total delivery time. It gives us a useful baseline before trying a second regression model on top of the three components.

In [ ]:
values_df["sum_total_delivery_duration"] = (
    values_df["prep_duration_prediction"]
    + values_df["estimated_store_to_consumer_driving_duration"]
    + values_df["estimated_order_place_duration"]
)

simple_sum_rmse = mean_squared_error(
    values_df["actual_total_delivery_duration"],
    values_df["sum_total_delivery_duration"]
) ** 0.5

print(f"RMSE from simply adding the three durations: {simple_sum_rmse:,.2f} seconds")
display(values_df.head())

The original solution reports an RMSE of approximately **2,036 seconds** for the simple sum, which is very close to the direct model. The next experiment is to let another regression model learn how the three components combine, rather than assuming that they should simply be added with equal weight.

## 15. Fit a second model to the duration components

The second model receives three inputs:

1. predicted preparation duration
2. estimated store to customer driving duration
3. estimated order placement duration

Its target is the actual total delivery duration.

The guided solution compares Linear Regression, Ridge, Decision Tree, Random Forest, XGBoost, LightGBM and an MLP Regressor at this stage.

In [ ]:
X_components = values_df[[
    "prep_duration_prediction",
    "estimated_store_to_consumer_driving_duration",
    "estimated_order_place_duration"
]]

y_total = values_df["actual_total_delivery_duration"]

X_comp_train, X_comp_test, y_comp_train, y_comp_test = train_test_split(
    X_components,
    y_total,
    test_size=0.2,
    random_state=42
)

component_models = {
    "LinearReg": linear_model.LinearRegression(),
    "Ridge": linear_model.Ridge(),
    "DecisionTree": tree.DecisionTreeRegressor(max_depth=6, random_state=42),
    "RandomForest": RandomForestRegressor(random_state=42),
    "XGBoost": XGBRegressor(random_state=42),
    "LGBM": LGBMRegressor(random_state=42),
    "MLP": MLPRegressor(random_state=42)
}

component_results = []

for model_name, model in component_models.items():
    fitted_model, predictions, train_rmse, test_rmse = fit_and_score_regression(
        X_comp_train,
        y_comp_train,
        X_comp_test,
        y_comp_test,
        model
    )

    component_results.append({
        "model": model_name,
        "train_RMSE": train_rmse,
        "test_RMSE": test_rmse
    })

component_results_df = pd.DataFrame(component_results).sort_values("test_RMSE")
display(component_results_df)

In the supplied solution, Linear Regression and Ridge Regression perform best at this stage, both reaching an RMSE of about **987 seconds**. This is a substantial improvement over predicting the full delivery duration directly.

The result suggests that separating preparation time from the other estimated activities gives the second model a more useful set of inputs.

## 16. Give a small neural network a try

Since the second stage uses only three numerical inputs, I also wanted to see whether a simple neural network could improve on the classical models.

The network is intentionally small: one hidden layer with 16 neurons and ReLU activation, followed by a single linear output. It is trained using SGD and mean squared error.

I'll scale the three inputs and the target, train for 100 epochs, and then convert the predictions back to seconds for a fair comparison.

In [ ]:
# Split first so the scalers only learn from the training portion.
X_ann_train_raw, X_ann_test_raw, y_ann_train_raw, y_ann_test_raw = train_test_split(
    X_components,
    y_total,
    test_size=0.2,
    random_state=42
)

ann_scaler_X = StandardScaler()
ann_scaler_y = StandardScaler()

X_ann_train = ann_scaler_X.fit_transform(X_ann_train_raw)
X_ann_test = ann_scaler_X.transform(X_ann_test_raw)

y_ann_train = ann_scaler_y.fit_transform(
    y_ann_train_raw.to_numpy().reshape(-1, 1)
)
y_ann_test = ann_scaler_y.transform(
    y_ann_test_raw.to_numpy().reshape(-1, 1)
)

def create_model(feature_set_size):
    """Build the small feedforward network used for this comparison."""
    model = Sequential([
        Input(shape=(feature_set_size,)),
        Dense(16, activation="relu"),
        Dense(1, activation="linear")
    ])

    model.compile(
        optimizer="sgd",
        loss="mse",
        metrics=[tf.keras.metrics.RootMeanSquaredError()]
    )

    return model

ann_model = create_model(X_ann_train.shape[1])

history = ann_model.fit(
    X_ann_train,
    y_ann_train,
    epochs=100,
    batch_size=64,
    verbose=1
)

ann_predictions_scaled = ann_model.predict(X_ann_test, verbose=0)
ann_predictions = ann_scaler_y.inverse_transform(ann_predictions_scaled).ravel()
y_ann_test_actual = ann_scaler_y.inverse_transform(y_ann_test).ravel()

ann_rmse = mean_squared_error(y_ann_test_actual, ann_predictions) ** 0.5
print(f"ANN test RMSE: {ann_rmse:,.2f} seconds")

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(history.history["loss"])
plt.title("Neural network training loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.tight_layout()
plt.show()

The supplied solution reports an ANN error of approximately **998 seconds**. That is close to the Linear Regression result, but it does not improve on it. Because this notebook fits the scalers using training data only, a fresh run can produce different scores from the original saved output.

For this problem, the simpler second stage model is a sensible choice. A more elaborate network might benefit from additional tuning, but the current results do not suggest that the extra complexity is necessary.

## 17. Wrap up

The first set of models predicts total delivery duration directly. Their RMSE values are generally around 2,030 to 2,060 seconds.

The two stage approach works better:

1. LightGBM predicts the preparation portion of the order.
2. A second regression model combines that prediction with the estimated driving and order placement durations.

The supplied guided solution reports the following comparison for the second stage:

| Model | RMSE (seconds) |
|---|---:|
| Linear Regression | 986.69 |
| Ridge Regression | 986.69 |
| MLP Regressor | 987.30 |
| Random Forest | 1,164.00 |
| LightGBM | 1,079.29 |
| XGBoost | 1,370.41 |
| Decision Tree | 1,235.58 |

The small ANN experiment reports an RMSE of approximately 997.89 seconds.

**Takeaway:** Decomposing the delivery into preparation, driving and order placement gives a much stronger result than predicting the full duration as one value. In this experiment, a relatively simple Linear Regression model on top of the LightGBM preparation estimate provides the best reported performance.

## A note before using this as a production model

This notebook follows the supplied guided solution and its experimental setup. The results are useful for comparing approaches on this dataset, but they are not a guarantee of real world performance. A production workflow would need a final untouched validation set, careful checks for information that would not be available at prediction time, and further tuning of the models and feature engineering.

The dataset file `historical_data.csv` is not included with this notebook. Add it to the same directory before running the cells from top to bottom.